In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ── Load all files ──────────────────────────────────────────────
personal   = pd.read_csv(r'D:\plfs_project\data\personal_data.csv')
household  = pd.read_csv(r'D:\plfs_project\data\household_data.csv')
districts  = pd.read_csv(r'D:\plfs_project\data\district_codes.csv')

print("✅ Files loaded successfully!")
print(f"   Personal data  : {personal.shape[0]:,} rows × {personal.shape[1]} columns")
print(f"   Household data : {household.shape[0]:,} rows × {household.shape[1]} columns")
print(f"   District codes : {districts.shape[0]:,} rows × {districts.shape[1]} columns")

✅ Files loaded successfully!
   Personal data  : 415,549 rows × 140 columns
   Household data : 101,957 rows × 38 columns
   District codes : 744 rows × 4 columns


In [4]:
# ── Sex ─────────────────────────────────────────────────────────
sex_map = {1: 'Male', 2: 'Female', 3: 'Transgender'}

# ── Rural / Urban ───────────────────────────────────────────────
stream_map = {1: 'Rural', 2: 'Rural', 3: 'Rural',
              4: 'Rural', 5: 'Urban', 6: 'Urban'}

# ── Education level ──────────────────────────────────────────────
edu_map = {
    1:  'Not Literate',
    2:  'Literate (No Schooling)',
    3:  'Literate (No Schooling)',
    4:  'Literate (No Schooling)',
    5:  'Below Primary',
    6:  'Primary',
    7:  'Middle',
    8:  'Secondary',
    10: 'Higher Secondary',
    11: 'Diploma',
    12: 'Graduate',
    13: 'Postgraduate & Above'
}

# ── Activity Status ──────────────────────────────────────────────
activity_map = {
    11: 'Self-Employed (Own Account)',
    12: 'Self-Employed (Employer)',
    21: 'Unpaid Family Worker',
    31: 'Regular Salaried/Wage',
    41: 'Casual Labour (Public Works)',
    51: 'Casual Labour (Other)',
    81: 'Unemployed (Seeking Work)',
    91: 'Student',
    92: 'Domestic Duties',
    93: 'Domestic + Subsistence Work',
    94: 'Pensioner/Rentier',
    95: 'Disabled',
    97: 'Other',
    99: 'Below 5 Years'
}

# ── State codes ──────────────────────────────────────────────────
state_map = dict(zip(districts['state_code'], districts['state_name']))

# ── Apply all mappings ───────────────────────────────────────────
personal['gender']          = personal['SEX'].map(sex_map)
personal['area_type']       = personal['STRM'].map(stream_map)
personal['education']       = personal['GEDU_LVL'].map(edu_map)
personal['activity_status'] = personal['PAS'].map(activity_map)
personal['state_name']      = personal['ST'].map(state_map)

print("✅ All columns decoded!")
print(personal[['gender','area_type','education','activity_status','state_name']].head(5))

✅ All columns decoded!
   gender area_type         education              activity_status  \
0    Male     Rural         Secondary        Regular Salaried/Wage   
1  Female     Rural         Secondary         Unpaid Family Worker   
2    Male     Rural  Higher Secondary  Self-Employed (Own Account)   
3  Female     Rural  Higher Secondary        Regular Salaried/Wage   
4  Female     Rural           Primary                      Student   

       state_name  
0  Andhra Pradesh  
1  Andhra Pradesh  
2  Andhra Pradesh  
3  Andhra Pradesh  
4  Andhra Pradesh  


In [5]:
# ── Keep only working-age population (15–60) ────────────────────
df = personal[(personal['AGE'] >= 15) & (personal['AGE'] <= 60)].copy()

# ── Drop rows where activity status is unknown ───────────────────
df = df[df['PAS'] != 99]

# ── Create Labour Force categories ──────────────────────────────
employed_codes   = [11, 12, 21, 31, 41, 51]
unemployed_codes = [81]
nilf_codes       = [91, 92, 93, 94, 95, 97]

def lf_category(code):
    if code in employed_codes:
        return 'Employed'
    elif code in unemployed_codes:
        return 'Unemployed'
    else:
        return 'Not in Labour Force'

df['lf_status'] = df['PAS'].apply(lf_category)

# ── Education order for charts ───────────────────────────────────
edu_order = ['Not Literate', 'Below Primary', 'Primary', 'Middle',
             'Secondary', 'Higher Secondary', 'Diploma',
             'Graduate', 'Postgraduate & Above']

print("✅ Data cleaned!")
print(f"   Working-age individuals : {len(df):,}")
print()
print(df['lf_status'].value_counts())

✅ Data cleaned!
   Working-age individuals : 277,485

lf_status
Employed               152726
Not in Labour Force    116067
Unemployed               8692
Name: count, dtype: int64


In [6]:
total        = len(df)
employed     = (df['lf_status'] == 'Employed').sum()
unemployed   = (df['lf_status'] == 'Unemployed').sum()
labour_force = employed + unemployed

lfpr = (labour_force / total) * 100
ur   = (unemployed / labour_force) * 100
wpr  = (employed / total) * 100

# Female specific
female_df   = df[df['gender'] == 'Female']
f_lf        = ((female_df['lf_status'] == 'Employed') | (female_df['lf_status'] == 'Unemployed')).sum()
female_lfpr = (f_lf / len(female_df)) * 100

# Youth (15-29)
youth_df = df[df['AGE'] <= 29]
y_lf     = ((youth_df['lf_status'] == 'Employed') | (youth_df['lf_status'] == 'Unemployed')).sum()
y_ur     = ((youth_df['lf_status'] == 'Unemployed').sum() / y_lf) * 100

print("=" * 50)
print("     INDIA PLFS 2024 — KEY INDICATORS")
print("=" * 50)
print(f"  Total Working-Age People  : {total:,}")
print(f"  Labour Force Part. Rate   : {lfpr:.1f}%")
print(f"  Worker Population Ratio   : {wpr:.1f}%")
print(f"  Unemployment Rate         : {ur:.1f}%")
print(f"  Female LFPR               : {female_lfpr:.1f}%")
print(f"  Youth Unemployment Rate   : {y_ur:.1f}%")
print("=" * 50)

     INDIA PLFS 2024 — KEY INDICATORS
  Total Working-Age People  : 277,485
  Labour Force Part. Rate   : 58.2%
  Worker Population Ratio   : 55.0%
  Unemployment Rate         : 5.4%
  Female LFPR               : 35.1%
  Youth Unemployment Rate   : 16.0%


In [7]:
#Education Paradox chart:

lf_df = df[df['lf_status'].isin(['Employed', 'Unemployed'])].copy()
lf_df = lf_df[lf_df['education'].isin(edu_order)]

edu_ur = (
    lf_df.groupby('education')
    .apply(lambda x: (x['lf_status'] == 'Unemployed').sum() / len(x) * 100)
    .reindex(edu_order)
    .reset_index()
)
edu_ur.columns = ['Education Level', 'Unemployment Rate (%)']

fig = px.bar(
    edu_ur,
    x='Education Level', y='Unemployment Rate (%)',
    title='📊 The Education Paradox — Higher Education = Higher Unemployment?',
    color='Unemployment Rate (%)',
    color_continuous_scale='Reds',
    text=edu_ur['Unemployment Rate (%)'].round(1)
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

In [8]:
state_gender = (
    df[df['state_name'].notna()]
    .groupby(['state_name', 'gender'])
    .apply(lambda x: ((x['lf_status'] == 'Employed') | (x['lf_status'] == 'Unemployed')).sum() / len(x) * 100)
    .reset_index()
)
state_gender.columns = ['State', 'Gender', 'LFPR (%)']

fig = px.bar(
    state_gender,
    x='State', y='LFPR (%)', color='Gender',
    barmode='group',
    title='👫 Male vs Female Labour Force Participation by State',
    color_discrete_map={'Male': '#2563EB', 'Female': '#EC4899'},
    text=state_gender['LFPR (%)'].round(0)
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [9]:
area_stats = (
    df[df['lf_status'].isin(['Employed','Unemployed'])]
    .groupby(['area_type','gender'])
    .apply(lambda x: (x['lf_status'] == 'Unemployed').sum() / len(x) * 100)
    .reset_index()
)
area_stats.columns = ['Area','Gender','Unemployment Rate (%)']

fig = px.bar(
    area_stats,
    x='Area', y='Unemployment Rate (%)', color='Gender',
    barmode='group',
    title='🏙️ Rural vs Urban Unemployment Rate by Gender',
    color_discrete_map={'Male': '#2563EB', 'Female': '#EC4899'},
    text=area_stats['Unemployment Rate (%)'].round(1)
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [10]:
women_nilf = df[
    (df['gender'] == 'Female') &
    (df['lf_status'] == 'Not in Labour Force')
]['activity_status'].value_counts().reset_index()
women_nilf.columns = ['Reason', 'Count']
women_nilf = women_nilf[women_nilf['Reason'] != 'Below 5 Years']

fig = px.pie(
    women_nilf,
    names='Reason', values='Count',
    title='👩 Why Are Women Not in the Labour Force?',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

In [11]:
export_cols = [
    'ST', 'state_name', 'DC', 'STRM', 'area_type',
    'SEX', 'gender', 'AGE', 'GEDU_LVL', 'education',
    'PAS', 'activity_status', 'lf_status',
    'ERN_REG', 'ERN_SELF', 'MULT'
]

clean_df = df[export_cols].copy()
clean_df['total_earnings'] = clean_df['ERN_REG'] + clean_df['ERN_SELF']
clean_df.to_csv(r'D:\plfs_project\data\plfs_clean.csv', index=False)

print("✅ Clean data exported!")
print(f"   Rows   : {len(clean_df):,}")
print(f"   Columns: {len(clean_df.columns)}")

✅ Clean data exported!
   Rows   : 277,485
   Columns: 17
